In [13]:
import pandas as pd
import numpy as np


train_df = pd.read_csv("../data/processed_data/train.csv")
test_df = pd.read_csv("../data/processed_data/test.csv")
#schedules_df = pd.read_csv("../data/processed_data/schedules.csv")

#print(len(schedules_df))

#schedules_df.head()


In [14]:

reference_start_time = pd.to_datetime("2024-01-01 00:00:00")
reference_end_time = pd.to_datetime("2025-01-01 00:00:00")  # Might need to adjust the end time
total_time_span = (reference_end_time - reference_start_time).total_seconds()

In [15]:

#schedules_df['sailing_time_converted'] = pd.to_datetime(schedules_df['sailingDate'], errors='coerce')
#schedules_df['arrival_time_converted'] = pd.to_datetime(schedules_df['arrivalDate'], errors='coerce')
#schedules_df = schedules_df.sort_values(by=['sailing_time_converted']).reset_index(drop=True)

# Might have to make this timezone aware!!
#schedules_df['sailing_time_converted'] = schedules_df['sailing_time_converted'].dt.tz_localize(None)
#schedules_df['arrival_time_converted'] = schedules_df['arrival_time_converted'].dt.tz_localize(None)

#schedules_df = schedules_df[schedules_df["sailing_time_converted"] <= reference_end_time]
#schedules_df = schedules_df[schedules_df["sailing_time_converted"] >= reference_start_time]
#schedules_df = schedules_df[schedules_df["sailing_time_converted"] < schedules_df["arrival_time_converted"]]


#schedules_df.drop(['sailing_time_converted'], axis=1, inplace=True)
#schedules_df.drop(['arrival_time_converted'], axis=1, inplace=True)


#schedules_df.tail()

In [16]:
# Training data - time feature conversion and normalization

train_df['original_time_converted'] = pd.to_datetime(train_df['time'], errors='coerce')
train_df['time_position_1_steps_ago_converted'] = pd.to_datetime(train_df['time_position_1_steps_ago'], errors='coerce')
train_df['time_position_2_steps_ago_converted'] = pd.to_datetime(train_df['time_position_2_steps_ago'], errors='coerce')
train_df['time_position_3_steps_ago_converted'] = pd.to_datetime(train_df['time_position_3_steps_ago'], errors='coerce')
train_df['time_position_4_steps_ago_converted'] = pd.to_datetime(train_df['time_position_4_steps_ago'], errors='coerce')
train_df['time_position_5_steps_ago_converted'] = pd.to_datetime(train_df['time_position_5_steps_ago'], errors='coerce')

train_df['time_cog_1_step_ago_converted'] = pd.to_datetime(train_df['time_cog_1_step_ago'], errors='coerce')
train_df['time_cog_2_steps_ago_converted'] = pd.to_datetime(train_df['time_cog_2_steps_ago'], errors='coerce')


train_df['hours_passed'] = train_df['original_time_converted'].diff().dt.total_seconds() / 3600
hours = train_df['original_time_converted'].dt.hour
minutes = train_df['original_time_converted'].dt.minute

# Convert hour to cyclical features
train_df['hour_sin'] = (np.sin(2 * np.pi * hours / 24) + 1) / 2
train_df['hour_cos'] = (np.cos(2 * np.pi * hours / 24) + 1) / 2

# Convert minute to cyclical features and normalize to [0,1]
train_df['minute_sin'] = (np.sin(2 * np.pi * minutes / 60) + 1) / 2
train_df['minute_cos'] = (np.cos(2 * np.pi * minutes / 60) + 1) / 2



train_df['time'] = (train_df['original_time_converted'] - reference_start_time).dt.total_seconds()
train_df['time_position_1_steps_ago'] = (train_df['time_position_1_steps_ago_converted'] - reference_start_time).dt.total_seconds()
train_df['time_position_2_steps_ago'] = (train_df['time_position_2_steps_ago_converted'] - reference_start_time).dt.total_seconds()
train_df['time_position_3_steps_ago'] = (train_df['time_position_3_steps_ago_converted'] - reference_start_time).dt.total_seconds()
train_df['time_position_4_steps_ago'] = (train_df['time_position_4_steps_ago_converted'] - reference_start_time).dt.total_seconds()
train_df['time_position_5_steps_ago'] = (train_df['time_position_5_steps_ago_converted'] - reference_start_time).dt.total_seconds()

train_df['time_cog_1_step_ago'] = (train_df['time_cog_1_step_ago_converted'] - reference_start_time).dt.total_seconds()
train_df['time_cog_2_steps_ago'] = (train_df['time_cog_2_steps_ago_converted'] - reference_start_time).dt.total_seconds()


# Normalize time (between 0 and 1)
train_df['time'] = train_df['time'] / total_time_span
train_df['time_position_1_steps_ago'] = train_df['time_position_1_steps_ago'] / total_time_span
train_df['time_position_2_steps_ago'] = train_df['time_position_2_steps_ago'] / total_time_span
train_df['time_position_3_steps_ago'] = train_df['time_position_3_steps_ago'] / total_time_span
train_df['time_position_4_steps_ago'] = train_df['time_position_4_steps_ago'] / total_time_span
train_df['time_position_5_steps_ago'] = train_df['time_position_5_steps_ago'] / total_time_span

train_df['time_cog_1_step_ago'] = train_df['time_cog_1_step_ago'] / total_time_span
train_df['time_cog_2_steps_ago'] = train_df['time_cog_2_steps_ago'] / total_time_span

train_df['hours_passed'] = train_df['hours_passed'] / 24

# Add new features
train_df['month_of_the_year'] = train_df['original_time_converted'].dt.month  # Month (1-12)
train_df['week_of_the_year'] = train_df['original_time_converted'].dt.isocalendar().week  # Week (1-53)
train_df['day_of_the_year'] = train_df['original_time_converted'].dt.dayofyear  # Day of year (1-365)
train_df['day_of_the_month'] = train_df['original_time_converted'].dt.day  # Day of month (1-31)
train_df['day_of_the_week'] = train_df['original_time_converted'].dt.dayofweek  # Day of week (0-6, where 0 is Monday)
train_df['hour_of_the_day'] = train_df['original_time_converted'].dt.hour  # Hour (0-23)

# Normalize features
# Min-max normalization to scale features between 0 and 1
train_df['month_of_the_year'] = (train_df['month_of_the_year'] - 1) / 11  # Normalize month (1-12)
train_df['week_of_the_year'] = (train_df['week_of_the_year'] - 1) / 52  # Normalize week (1-53)
train_df['day_of_the_year'] = (train_df['day_of_the_year'] - 1) / 365  # Normalize day of year (1-365)
train_df['day_of_the_month'] = (train_df['day_of_the_month'] - 1) / 30  # Normalize day of month (1-31)
train_df['day_of_the_week'] = train_df['day_of_the_week'] / 6  # Normalize day of week (0-6)
train_df['hour_of_the_day'] = train_df['hour_of_the_day'] / 23  # Normalize hour (0-23)

# Add time diff features
train_df = train_df.sort_values(['vesselId', 'original_time_converted'])
train_df['time_diff'] = train_df.groupby('vesselId')['original_time_converted'].diff(-1)  # Using -1 for difference with next row

# Create boolean features for different time thresholds
train_df['time_diff_gt_10min'] = ((-train_df['time_diff'].dt.total_seconds()) > 10 * 60).astype(int)
train_df['time_diff_gt_20min'] = ((-train_df['time_diff'].dt.total_seconds()) > 20 * 60).astype(int)
train_df['time_diff_gt_40min'] = ((-train_df['time_diff'].dt.total_seconds()) > 40 * 60).astype(int)
train_df['time_diff_gt_1hour'] = ((-train_df['time_diff'].dt.total_seconds()) > 1 * 3600).astype(int)
train_df['time_diff_gt_2hours'] = ((-train_df['time_diff'].dt.total_seconds()) > 2 * 3600).astype(int)
train_df['time_diff_gt_6hours'] = ((-train_df['time_diff'].dt.total_seconds()) > 6 * 3600).astype(int)
train_df['time_diff_gt_12hours'] = ((-train_df['time_diff'].dt.total_seconds()) > 12 * 3600).astype(int)
train_df['time_diff_gt_1day'] = ((-train_df['time_diff'].dt.total_seconds()) > 24 * 3600).astype(int)

# Fill NA values (last row of each vessel group) with 0
time_diff_columns = [col for col in train_df.columns if col.startswith('time_diff_gt_')]
train_df[time_diff_columns] = train_df[time_diff_columns].fillna(0)

# Drop the temporary time_diff column
train_df.drop('time_diff', axis=1, inplace=True)



# Drop intermediate columns
train_df.drop(
    [
        'original_time_converted',
        'time_position_1_steps_ago_converted',
        'time_position_2_steps_ago_converted',
        'time_position_3_steps_ago_converted',
        'time_position_4_steps_ago_converted',
        'time_position_5_steps_ago_converted',
        'time_cog_1_step_ago_converted',
        'time_cog_2_steps_ago_converted'
    ], 
    axis=1,
    inplace=True
)

train_df.head()



,time,cog,sog,rot,heading,navstat,etaRaw,latitude,longitude,vesselId,...,day_of_the_week,hour_of_the_day,time_diff_gt_10min,time_diff_gt_20min,time_diff_gt_40min,time_diff_gt_1hour,time_diff_gt_2hours,time_diff_gt_6hours,time_diff_gt_12hours,time_diff_gt_1day
0,0.031663,0.858217,17.1,-6,316,0,01-08 06:00,7.50361,77.58340,61e9f38eb937134a3c4bfd8b,...,0.666667,0.608696,1,1,0,0,0,0,0,0
1,0.031707,0.856825,17.3,5,313,0,01-14 23:30,7.57302,77.49505,61e9f38eb937134a3c4bfd8b,...,0.666667,0.608696,1,1,0,0,0,0,0,0
2,0.031757,0.854596,16.9,5,312,0,01-14 23:30,7.65043,77.39404,61e9f38eb937134a3c4bfd8b,...,0.666667,0.608696,1,1,0,0,0,0,0,0
3,0.031798,0.857660,16.9,6,313,0,01-14 23:30,7.71275,77.31394,61e9f38eb937134a3c4bfd8b,...,0.666667,0.652174,1,1,0,0,0,0,0,0
4,0.031838,0.855153,16.3,7,313,0,01-14 23:30,7.77191,77.23585,61e9f38eb937134a3c4bfd8b,...,0.666667,0.652174,1,0,0,0,0,0,0,0


In [17]:
# Test data - time feature conversion and normalization
test_df['time_converted'] = pd.to_datetime(test_df['time'], errors='coerce')
test_df['time'] = (test_df['time_converted'] - reference_start_time).dt.total_seconds()
test_df['time'] = test_df['time'] / total_time_span

# Add new features
test_df['month_of_the_year'] = test_df['time_converted'].dt.month  # Month (1-12)
test_df['week_of_the_year'] = test_df['time_converted'].dt.isocalendar().week  # Week (1-53)
test_df['day_of_the_year'] = test_df['time_converted'].dt.dayofyear  # Day of year (1-365)
test_df['day_of_the_month'] = test_df['time_converted'].dt.day  # Day of month (1-31)
test_df['day_of_the_week'] = test_df['time_converted'].dt.dayofweek  # Day of week (0-6, where 0 is Monday)
test_df['hour_of_the_day'] = test_df['time_converted'].dt.hour  # Hour (0-23)

# Normalize features
# Min-max normalization to scale features between 0 and 1
test_df['month_of_the_year'] = (test_df['month_of_the_year'] - 1) / 11  # Normalize month (1-12)
test_df['week_of_the_year'] = (test_df['week_of_the_year'] - 1) / 52  # Normalize week (1-53)
test_df['day_of_the_year'] = (test_df['day_of_the_year'] - 1) / 365  # Normalize day of year (1-365)
test_df['day_of_the_month'] = (test_df['day_of_the_month'] - 1) / 30  # Normalize day of month (1-31)
test_df['day_of_the_week'] = test_df['day_of_the_week'] / 6  # Normalize day of week (0-6)
test_df['hour_of_the_day'] = test_df['hour_of_the_day'] / 23  # Normalize hour (0-23)

test_df['hours_passed'] = test_df['time_converted'].diff().dt.total_seconds() / 3600
hours = test_df['time_converted'].dt.hour
minutes = test_df['time_converted'].dt.minute

# Convert hour to cyclical features
test_df['hour_sin'] = (np.sin(2 * np.pi * hours / 24) + 1) / 2
test_df['hour_cos'] = (np.cos(2 * np.pi * hours / 24) + 1) / 2

# Convert minute to cyclical features and normalize to [0,1]
test_df['minute_sin'] = (np.sin(2 * np.pi * minutes / 60) + 1) / 2
test_df['minute_cos'] = (np.cos(2 * np.pi * minutes / 60) + 1) / 2

test_df['hours_passed'] = test_df['hours_passed'] / 24

# Add time diff features
test_df = test_df.sort_values(['vesselId', 'time_converted'])
test_df['time_diff'] = test_df.groupby('vesselId')['time_converted'].diff(-1)  # Using -1 for difference with next row

# Create boolean features for different time thresholds
test_df['time_diff_gt_10min'] = ((-test_df['time_diff'].dt.total_seconds()) > 10 * 60).astype(int)
test_df['time_diff_gt_20min'] = ((-test_df['time_diff'].dt.total_seconds()) > 20 * 60).astype(int)
test_df['time_diff_gt_40min'] = ((-test_df['time_diff'].dt.total_seconds()) > 40 * 60).astype(int)
test_df['time_diff_gt_1hour'] = ((-test_df['time_diff'].dt.total_seconds()) > 1 * 3600).astype(int)
test_df['time_diff_gt_2hours'] = ((-test_df['time_diff'].dt.total_seconds()) > 2 * 3600).astype(int)
test_df['time_diff_gt_6hours'] = ((-test_df['time_diff'].dt.total_seconds()) > 6 * 3600).astype(int)
test_df['time_diff_gt_12hours'] = ((-test_df['time_diff'].dt.total_seconds()) > 12 * 3600).astype(int)
test_df['time_diff_gt_1day'] = ((-test_df['time_diff'].dt.total_seconds()) > 24 * 3600).astype(int)

# Fill NA values (last row of each vessel group) with 0
time_diff_columns = [col for col in test_df.columns if col.startswith('time_diff_gt_')]
test_df[time_diff_columns] = test_df[time_diff_columns].fillna(0)

# Drop the temporary time_diff column
test_df.drop('time_diff', axis=1, inplace=True)



# Drop intermediate columns
test_df.drop(['time_converted'], axis=1, inplace=True)

test_df.head()

,ID,vesselId,time,scaling_factor,port_lat,port_long,month_of_the_year,week_of_the_year,day_of_the_year,day_of_the_month,...,minute_sin,minute_cos,time_diff_gt_10min,time_diff_gt_20min,time_diff_gt_40min,time_diff_gt_1hour,time_diff_gt_2hours,time_diff_gt_6hours,time_diff_gt_12hours,time_diff_gt_1day
4,4,61e9f38eb937134a3c4bfd8d,0.349750,0.3,NaN,NaN,0.363636,0.346154,0.350685,0.233333,...,0.975528,0.654508,1,1,0,0,0,0,0,0
201,201,61e9f38eb937134a3c4bfd8d,0.349802,0.3,NaN,NaN,0.363636,0.346154,0.350685,0.233333,...,0.095492,0.206107,1,1,1,0,0,0,0,0
583,583,61e9f38eb937134a3c4bfd8d,0.349904,0.3,NaN,NaN,0.363636,0.346154,0.350685,0.233333,...,0.345492,0.024472,1,0,0,0,0,0,0,0
701,701,61e9f38eb937134a3c4bfd8d,0.349938,0.3,NaN,NaN,0.363636,0.346154,0.350685,0.233333,...,0.095492,0.793893,1,0,0,0,0,0,0,0
829,829,61e9f38eb937134a3c4bfd8d,0.349961,0.3,NaN,NaN,0.363636,0.346154,0.350685,0.233333,...,0.654508,0.975528,1,1,0,0,0,0,0,0


In [18]:
# Schedule data - time feature conversion and normalization
#schedules_df['sailing_time_converted'] = pd.to_datetime(schedules_df['sailingDate'], errors='coerce').dt.tz_localize(None) # Might have to make this timezone aware!!
#schedules_df['arrival_time_converted'] = pd.to_datetime(schedules_df['arrivalDate'], errors='coerce').dt.tz_localize(None) # Might have to make this timezone aware!!
#schedules_df['sailingDate'] = (schedules_df['sailing_time_converted'] - reference_start_time).dt.total_seconds()
#schedules_df['arrivalDate'] = (schedules_df['arrival_time_converted'] - reference_start_time).dt.total_seconds()

#schedules_df['sailingDate'] = schedules_df['sailingDate'] / total_time_span
#schedules_df['arrivalDate'] = schedules_df['arrivalDate'] / total_time_span


# Add new features
#schedules_df['sailing_week_of_the_year'] = schedules_df['sailing_time_converted'].dt.isocalendar().week  # Week number of the year
#schedules_df['sailing_day_of_the_year'] = schedules_df['sailing_time_converted'].dt.dayofyear
#schedules_df['arrival_week_of_the_year'] = schedules_df['arrival_time_converted'].dt.isocalendar().week  # Week number of the year
#schedules_df['arrival_day_of_the_year'] = schedules_df['arrival_time_converted'].dt.dayofyear

# Normalize other features in test data
#schedules_df['sailing_week_of_the_year'] = (schedules_df['sailing_week_of_the_year'] - 1) / 52  # Normalize week_of_the_year (1-53)
#schedules_df['sailing_day_of_the_year'] = (schedules_df['sailing_day_of_the_year'] - 1) / 365  # Normalize day_of_the_year (1-365)
#schedules_df['arrival_week_of_the_year'] = (schedules_df['arrival_week_of_the_year'] - 1) / 52  # Normalize week_of_the_year (1-53)
#schedules_df['arrival_day_of_the_year'] = (schedules_df['arrival_day_of_the_year'] - 1) / 365  # Normalize day_of_the_year (1-365)

# Drop intermediate columns
#schedules_df.drop(['sailing_time_converted'], axis=1, inplace=True)
#schedules_df.drop(['arrival_time_converted'], axis=1, inplace=True)

#schedules_df.head()


In [19]:
train_df.to_csv('../data/processed_data/train.csv', index=False)
test_df.to_csv("../data/processed_data/test.csv", index=False)
#schedules_df.to_csv("../data/processed_data/schedules.csv", index=False)